# Dinámica estacional de los humedales de El Yali

Análisis exploratorio de indicadores ambientales para la Albufera, Laguna Colejuda y Laguna Matanza durante 2019–2025. El cuaderno utiliza tablas analíticas derivadas de composiciones estacionales previamente validadas y se concentra en el control de calidad, la comparación temporal y la comunicación de resultados.

## Alcance

La superficie estimada representa píxeles con respuesta espectral de agua abierta dentro de sectores analíticos. No corresponde a una delimitación legal, una medición oficial ni una estimación completa de ambientes húmedos cubiertos por vegetación. Las relaciones con precipitación son exploratorias y no implican causalidad.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd
import seaborn as sns

sns.set_theme(style="white", context="notebook")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "axes.edgecolor": "#303030",
    "axes.labelcolor": "#222222",
    "xtick.color": "#222222",
    "ytick.color": "#222222",
    "text.color": "#222222",
})

def preparar_eje(eje):
    """Eje sobrio: fondo blanco, marco completo y sin cuadrícula interna."""
    eje.grid(False)
    for lado in ("top", "right", "bottom", "left"):
        eje.spines[lado].set_visible(True)
        eje.spines[lado].set_color("#303030")
        eje.spines[lado].set_linewidth(0.85)
    eje.tick_params(axis="both", width=0.75, length=3.2, color="#303030")

pd.set_option("display.max_columns", None)


In [ ]:
# El cuaderno funciona desde la raíz del repositorio o desde notebooks/.
candidatos = [Path.cwd(), Path.cwd().parent]
raiz = next(
    ruta for ruta in candidatos
    if (ruta / "data" / "results" / "sector_analysis").exists()
)

carpeta_datos = raiz / "data" / "results" / "sector_analysis"

serie = pd.read_csv(carpeta_datos / "serie_integrada_por_sector.csv")
sensibilidad = pd.read_csv(carpeta_datos / "sensibilidad_mndwi_por_sector.csv")
superficies = pd.read_csv(carpeta_datos / "superficies_sectores_analiticos.csv")

print(f"Registros cargados: {len(serie)}")
serie.head()

## Control de calidad

Se espera una observación por combinación de zona, año y temporada: tres unidades, siete años y dos temporadas, equivalentes a 42 registros.

In [ ]:
claves = ["zona", "anio", "temporada"]
columnas_analiticas = [
    "superficie_agua_ha",
    "ndvi_medio",
    "precipitacion_mm",
]

control_calidad = pd.Series({
    "registros": len(serie),
    "duplicados": int(serie.duplicated(claves).sum()),
    "valores_ausentes": int(serie[columnas_analiticas].isna().sum().sum()),
    "zonas": serie["zona"].nunique(),
    "anios": serie["anio"].nunique(),
    "temporadas": serie["temporada"].nunique(),
})

assert control_calidad["registros"] == 42
assert control_calidad["duplicados"] == 0
assert control_calidad["valores_ausentes"] == 0
assert serie["ndvi_medio"].between(-1, 1).all()
assert (serie["superficie_agua_ha"] >= 0).all()

control_calidad.to_frame("resultado")

## Indicadores comparables

La superficie de agua se normaliza por el tamaño de cada sector para evitar que la comparación dependa solamente de su extensión.

In [ ]:
datos = serie.merge(
    superficies,
    on="zona",
    how="left",
    validate="many_to_one",
    suffixes=("", "_referencia"),
)

if "superficie_sector_ha_referencia" in datos.columns:
    datos["superficie_sector_ha"] = datos["superficie_sector_ha"].fillna(
        datos["superficie_sector_ha_referencia"]
    )
    datos = datos.drop(columns="superficie_sector_ha_referencia")

datos["agua_abierta_pct"] = (
    datos["superficie_agua_ha"] / datos["superficie_sector_ha"] * 100
)

resumen = (
    datos.groupby(["zona", "temporada"], as_index=False)
    .agg(
        agua_promedio_ha=("superficie_agua_ha", "mean"),
        agua_maxima_ha=("superficie_agua_ha", "max"),
        cobertura_agua_promedio_pct=("agua_abierta_pct", "mean"),
        ndvi_promedio=("ndvi_medio", "mean"),
    )
)

resumen.round(2)

In [ ]:
orden_zonas = ["Albufera", "Laguna Colejuda", "Laguna Matanza"]
colores = {"invierno": "#2166AC", "verano": "#C86B2B"}

fig, ejes = plt.subplots(1, 3, figsize=(16, 4.6), sharex=True)

for eje, zona in zip(ejes, orden_zonas):
    subconjunto = datos.loc[datos["zona"] == zona]
    for temporada in ("invierno", "verano"):
        serie_zona = subconjunto.loc[subconjunto["temporada"] == temporada].sort_values("anio")
        eje.plot(
            serie_zona["anio"], serie_zona["superficie_agua_ha"],
            color=colores[temporada], linewidth=2.1, marker="o", markersize=4.5,
            label=temporada.capitalize(),
        )
    eje.set(title=zona, xlabel="Año")
    if zona == "Albufera":
        eje.set_ylabel("Agua abierta estimada (ha)")
    eje.set_xticks(range(2019, 2026))
    preparar_eje(eje)

manijas, etiquetas = ejes[0].get_legend_handles_labels()
fig.legend(manijas, etiquetas, title="Temporada", loc="upper center", ncol=2, frameon=False)
fig.suptitle("Variación estacional del agua abierta", x=0.01, ha="left", y=1.04, fontweight="bold")
fig.text(0.01, -0.02, "Azul: invierno; naranjo: verano. Cada panel tiene su propia escala vertical.", fontsize=9, color="#555555")
plt.tight_layout(rect=(0, 0.03, 1, 0.90))
plt.show()


La Albufera presenta agua abierta durante todas las temporadas analizadas. Colejuda muestra una contracción estival marcada, mientras Matanza concentra sus principales pulsos en los inviernos de 2020 y 2024.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.6), sharex=True, sharey=True)

for eje, zona in zip(ejes, orden_zonas):
    subconjunto = datos.loc[datos["zona"] == zona]
    for temporada in ("invierno", "verano"):
        serie_zona = subconjunto.loc[subconjunto["temporada"] == temporada].sort_values("anio")
        eje.plot(
            serie_zona["anio"], serie_zona["ndvi_medio"],
            color=colores[temporada], linewidth=2.1, marker="o", markersize=4.5,
            label=temporada.capitalize(),
        )
    eje.set(title=zona, xlabel="Año")
    if zona == "Albufera":
        eje.set_ylabel("NDVI medio")
    eje.set_xticks(range(2019, 2026))
    preparar_eje(eje)

manijas, etiquetas = ejes[0].get_legend_handles_labels()
fig.legend(manijas, etiquetas, title="Temporada", loc="upper center", ncol=2, frameon=False)
fig.suptitle("Condición espectral media de la vegetación", x=0.01, ha="left", y=1.04, fontweight="bold")
fig.text(0.01, -0.02, "Azul: invierno; naranjo: verano. NDVI es un indicador espectral y no equivale por sí solo a condición ecológica.", fontsize=9, color="#555555")
plt.tight_layout(rect=(0, 0.03, 1, 0.90))
plt.show()


El NDVI más alto de las lagunas interiores indica que la ausencia de píxeles clasificados como agua abierta no equivale necesariamente a la ausencia de condiciones húmedas. La vegetación emergente, el agua somera y los sedimentos modifican la respuesta espectral.

## Relación exploratoria con precipitación

Las correlaciones se calculan por zona y temporada. Cada coeficiente se basa en siete observaciones, por lo que se utiliza como descripción del periodo y no como evidencia causal.

In [ ]:
filas_correlacion = []

for (zona, temporada), grupo in datos.groupby(["zona", "temporada"]):
    filas_correlacion.append({
        "zona": zona,
        "temporada": temporada,
        "observaciones": len(grupo),
        "pearson_agua_precipitacion": grupo["superficie_agua_ha"].corr(
            grupo["precipitacion_mm"], method="pearson"
        ),
        "spearman_agua_precipitacion": grupo["superficie_agua_ha"].corr(
            grupo["precipitacion_mm"], method="spearman"
        ),
    })

correlaciones = pd.DataFrame(filas_correlacion)

correlaciones.round(3)

In [ ]:
invierno = datos.loc[datos["temporada"] == "invierno"]
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.6), sharex=True)

for eje, zona in zip(ejes, orden_zonas):
    subconjunto = invierno.loc[invierno["zona"] == zona]
    sns.regplot(
        data=subconjunto,
        x="precipitacion_mm",
        y="superficie_agua_ha",
        ci=None,
        scatter_kws={"s": 60, "color": "#2166AC", "edgecolor": "white", "linewidth": 0.8},
        line_kws={"color": "#4D4D4D", "linewidth": 1.3},
        ax=eje,
    )
    for fila in subconjunto.itertuples():
        eje.annotate(str(fila.anio), (fila.precipitacion_mm, fila.superficie_agua_ha), xytext=(4, 4), textcoords="offset points", fontsize=8)
    r = subconjunto["superficie_agua_ha"].corr(subconjunto["precipitacion_mm"])
    eje.set(title=f"{zona}\nPearson r = {r:.2f}; n = 7", xlabel="Precipitación CHIRPS (mm)")
    if zona == "Albufera":
        eje.set_ylabel("Agua abierta estimada (ha)")
    preparar_eje(eje)

fig.suptitle("Precipitación invernal y agua abierta", x=0.01, ha="left", y=1.04, fontweight="bold")
fig.text(0.01, -0.02, "Cada punto representa un año; la recta resume una asociación exploratoria, no causalidad.", fontsize=9, color="#555555")
plt.tight_layout(rect=(0, 0.03, 1, 0.92))
plt.show()


Colejuda presenta la asociación invernal más consistente. En Matanza, el coeficiente está influido por los pulsos de 2020 y 2024; en la Albufera, la precipitación no explica por sí sola la variación observada.

## Sensibilidad del umbral MNDWI

La estimación principal utiliza MNDWI mayor que cero. El gráfico siguiente compara tres umbrales: celeste = −0,1; azul = 0,0 (escenario principal); verde = 0,1. Las barras expresan el promedio de agua abierta estimada, no tres tipos de cobertura.


In [ ]:
columnas_umbral = ["-0.1", "0.0", "0.1"]
sensibilidad_resumen = sensibilidad.groupby("zona")[columnas_umbral].mean().reindex(orden_zonas)

colores_umbral = ["#9FC5D1", "#2166AC", "#7FAF9B"]
etiquetas_umbral = ["−0,1", "0,0\nprincipal", "0,1"]
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=False)

for eje, zona in zip(ejes, orden_zonas):
    valores = sensibilidad_resumen.loc[zona, columnas_umbral].to_numpy()
    barras = eje.bar(etiquetas_umbral, valores, color=colores_umbral, width=0.62, edgecolor="#303030", linewidth=0.55)
    for barra, valor in zip(barras, valores):
        eje.text(barra.get_x() + barra.get_width()/2, valor, f"{valor:.1f}", ha="center", va="bottom", fontsize=9)
    eje.set(title=zona)
    if zona == "Albufera":
        eje.set_ylabel("Agua abierta media (ha)")
    preparar_eje(eje)

leyenda = [Patch(facecolor=c, edgecolor="#303030", label=f"MNDWI {u}".replace(".", ",")) for c, u in zip(colores_umbral, ["-0.1", "0.0", "0.1"])]
fig.legend(handles=leyenda, title="Umbral aplicado", loc="upper center", ncol=3, frameon=False)
fig.suptitle("Sensibilidad de la superficie de agua al umbral MNDWI", x=0.01, ha="left", y=1.06, fontweight="bold")
fig.text(0.01, -0.02, "Promedio de 14 composiciones estacionales por sector (2019-2025). El umbral 0,0 se utiliza como escenario principal.", fontsize=9, color="#555555")
plt.tight_layout(rect=(0, 0.03, 1, 0.88))
plt.show()

sensibilidad_resumen.round(2)


## Conclusiones

1. El análisis agregado no representa adecuadamente la diversidad interna de El Yali.
2. La Albufera mantiene agua abierta con mayor persistencia que las lagunas interiores.
3. Colejuda responde con claridad a la estacionalidad de las precipitaciones.
4. Matanza alterna pulsos de agua abierta con periodos dominados por vegetación y otras coberturas húmedas.
5. La combinación de MNDWI, NDVI, precipitación y validación visual ofrece una interpretación más sólida que el uso aislado de un único indicador.